<a href="https://colab.research.google.com/github/aimldstejas/aibits-genai-notebooks/blob/main/course-1-deep-learning/lab-08-synthetic-defects-evaluated-honestly.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lab 8 (graded) — Synthetic defects, evaluated honestly
**Course 1: Hands-On Deep Learning with Python — Chapter 8: Generative models I (GANs)**

**Problem brief (Leo Farkas, Cobalt Manufacturing):** "Our rarest defect type has only
~200 images. Can we generate realistic extra examples — and should we trust them?"

**What you'll submit:** a DCGAN trained on Fashion-MNIST (fast, stable — proves the
mechanism), FID tracked over training, a sample gallery, then the augmentation experiment on
a Chapter-5-style detector with an honestly reported result (a negative result, well
analyzed, earns full marks).

In [ ]:
!pip install -q torchmetrics[image]

## 1. Data: Fashion-MNIST (always available via torchvision — no offline fallback needed)

In [ ]:
import torch
import torchvision
import torchvision.transforms as T
from torch.utils.data import DataLoader

torch.manual_seed(0)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
IMG_SIZE = 32

transform = T.Compose([T.Resize(IMG_SIZE), T.ToTensor(), T.Normalize([0.5], [0.5])])
fmnist = torchvision.datasets.FashionMNIST(root='./data', train=True, download=True, transform=transform)
loader = DataLoader(fmnist, batch_size=128, shuffle=True, drop_last=True)
print('Fashion-MNIST:', len(fmnist), 'images')

## 2. DCGAN

In [ ]:
import torch.nn as nn

Z_DIM = 100

class Generator(nn.Module):
    def __init__(self, z_dim=Z_DIM):
        super().__init__()
        self.net = nn.Sequential(
            nn.ConvTranspose2d(z_dim, 128, 4, 1, 0), nn.BatchNorm2d(128), nn.ReLU(True),   # 1x1 -> 4x4
            nn.ConvTranspose2d(128, 64, 4, 2, 1), nn.BatchNorm2d(64), nn.ReLU(True),        # 4x4 -> 8x8
            nn.ConvTranspose2d(64, 32, 4, 2, 1), nn.BatchNorm2d(32), nn.ReLU(True),         # 8x8 -> 16x16
            nn.ConvTranspose2d(32, 1, 4, 2, 1), nn.Tanh(),                                    # 16x16 -> 32x32
        )
    def forward(self, z):
        return self.net(z.view(z.size(0), Z_DIM, 1, 1))


class Discriminator(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(1, 32, 4, 2, 1), nn.LeakyReLU(0.2, True),                              # 32x32 -> 16x16
            nn.Conv2d(32, 64, 4, 2, 1), nn.BatchNorm2d(64), nn.LeakyReLU(0.2, True),          # 16x16 -> 8x8
            nn.Conv2d(64, 128, 4, 2, 1), nn.BatchNorm2d(128), nn.LeakyReLU(0.2, True),        # 8x8 -> 4x4
            nn.Conv2d(128, 1, 4, 1, 0), nn.Flatten(),                                          # 4x4 -> 1x1
        )
    def forward(self, x):
        return self.net(x)

G, D = Generator().to(device), Discriminator().to(device)
opt_G = torch.optim.Adam(G.parameters(), lr=2e-4, betas=(0.5, 0.999))
opt_D = torch.optim.Adam(D.parameters(), lr=2e-4, betas=(0.5, 0.999))
bce = nn.BCEWithLogitsLoss()

## 3. Train, tracking FID every few epochs

In [ ]:
from torchmetrics.image.fid import FrechetInceptionDistance
import matplotlib.pyplot as plt

def to_uint8_rgb(x):
    # FID expects uint8 RGB; Fashion-MNIST is grayscale in [-1,1] — denormalize and repeat channels
    x = ((x + 1) / 2 * 255).clamp(0, 255).to(torch.uint8)
    return x.repeat(1, 3, 1, 1)

N_EPOCHS = 15
fid_scores, sample_grids = [], []
fixed_noise = torch.randn(16, Z_DIM, device=device)

for epoch in range(N_EPOCHS):
    for real, _ in loader:
        real = real.to(device)
        bs = real.size(0)

        # --- discriminator step ---
        z = torch.randn(bs, Z_DIM, device=device)
        fake = G(z).detach()
        opt_D.zero_grad()
        loss_D = bce(D(real), torch.full((bs, 1), 0.9, device=device)) + \
                  bce(D(fake), torch.zeros(bs, 1, device=device))  # 0.9 = one-sided label smoothing
        loss_D.backward(); opt_D.step()

        # --- generator step (non-saturating loss) ---
        z = torch.randn(bs, Z_DIM, device=device)
        fake = G(z)
        opt_G.zero_grad()
        loss_G = bce(D(fake), torch.ones(bs, 1, device=device))
        loss_G.backward(); opt_G.step()

    if epoch % 3 == 0 or epoch == N_EPOCHS - 1:
        fid = FrechetInceptionDistance(feature=64, normalize=False).to(device)
        real_batch, _ = next(iter(loader))
        fid.update(to_uint8_rgb(real_batch.to(device)), real=True)
        with torch.no_grad():
            fake_batch = G(torch.randn(real_batch.size(0), Z_DIM, device=device))
        fid.update(to_uint8_rgb(fake_batch), real=False)
        score = float(fid.compute())
        fid_scores.append((epoch, score))
        print(f'epoch {epoch}: loss_D={loss_D.item():.3f} loss_G={loss_G.item():.3f} FID={score:.1f}')
        with torch.no_grad():
            sample_grids.append((epoch, G(fixed_noise).cpu()))

In [ ]:
epochs, scores = zip(*fid_scores)
plt.plot(epochs, scores, marker='o')
plt.xlabel('epoch'); plt.ylabel('FID (lower is better)'); plt.title('DCGAN — FID over training')
plt.show()

fig, axes = plt.subplots(1, len(sample_grids), figsize=(3 * len(sample_grids), 3))
for ax, (ep, grid) in zip(axes, sample_grids):
    img = torchvision.utils.make_grid(grid, nrow=4, normalize=True).permute(1, 2, 0)
    ax.imshow(img); ax.set_title(f'epoch {ep}'); ax.axis('off')
plt.suptitle('Sample gallery across training checkpoints')
plt.show()

## 4. The augmentation experiment
Does adding GAN-generated samples of a scarce class actually help a downstream detector, or
does it teach the detector to recognize generator artifacts instead? Train a small classifier
on Chapter 5's casting-style task twice — with and without GAN-augmented minority-class
images — and compare. This reuses Chapter 5's synthetic offline-fallback image generator so
the experiment is self-contained here; swap in your real Chapter 5 data/model if you have it.

In [ ]:
import numpy as np
from sklearn.metrics import recall_score

rng = np.random.default_rng(1)
IMG = 32

def make_casting_like(n, defect_rate, rng):
    X, y = [], []
    for i in range(n):
        label = int(rng.random() < defect_rate)
        base = rng.normal(0.5, 0.08, (IMG, IMG)).astype(np.float32)
        if label == 1:
            yy, xx = np.mgrid[0:IMG, 0:IMG]
            cy, cx = rng.integers(8, IMG - 8, 2)
            r = np.sqrt((yy - cy) ** 2 + (xx - cx) ** 2)
            base = np.clip(base + np.exp(-((r - 6) ** 2) / 5.0) * 0.5, 0, 1)
        X.append(base); y.append(label)
    return np.array(X), np.array(y)

# A realistically scarce minority class: only ~3% defective in the training pool
X_train_imb, y_train_imb = make_casting_like(1500, defect_rate=0.03, rng=rng)
X_val_c, y_val_c = make_casting_like(400, defect_rate=0.15, rng=rng)  # eval set, not imbalanced to near-zero

def train_small_classifier(X, y, n_epochs=12):
    Xt = torch.tensor(X, dtype=torch.float32).unsqueeze(1)
    yt = torch.tensor(y, dtype=torch.long)
    model = nn.Sequential(
        nn.Conv2d(1, 16, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
        nn.Conv2d(16, 32, 3, padding=1), nn.ReLU(), nn.AdaptiveAvgPool2d(1),
        nn.Flatten(), nn.Linear(32, 2),
    ).to(device)
    weight = torch.tensor([1.0, (y == 0).sum() / max(1, (y == 1).sum())], dtype=torch.float32).to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=1e-3)
    loss_fn = nn.CrossEntropyLoss(weight=weight)
    loader = DataLoader(torch.utils.data.TensorDataset(Xt, yt), batch_size=32, shuffle=True)
    for _ in range(n_epochs):
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            opt.zero_grad(); loss = loss_fn(model(xb), yb); loss.backward(); opt.step()
    return model

def eval_recall(model, X, y):
    model.eval()
    with torch.no_grad():
        preds = model(torch.tensor(X, dtype=torch.float32).unsqueeze(1).to(device)).argmax(1).cpu().numpy()
    return recall_score(y, preds, pos_label=1, zero_division=0)

# WITHOUT augmentation
model_no_aug = train_small_classifier(X_train_imb, y_train_imb)
recall_no_aug = eval_recall(model_no_aug, X_val_c, y_val_c)

# WITH augmentation — pad the minority class with more synthetic defective examples
n_extra = 200
X_extra, y_extra = make_casting_like(n_extra, defect_rate=1.0, rng=rng)
X_aug = np.concatenate([X_train_imb, X_extra]); y_aug = np.concatenate([y_train_imb, y_extra])
model_aug = train_small_classifier(X_aug, y_aug)
recall_aug = eval_recall(model_aug, X_val_c, y_val_c)

print(f'Recall for the defective class, WITHOUT augmentation: {recall_no_aug:.3f}')
print(f'Recall for the defective class, WITH augmentation:    {recall_aug:.3f}')
print(f'Effect: {recall_aug - recall_no_aug:+.3f}')

## 5. Report the effect honestly (fill in)
State the measured effect with a confidence interval if you re-run with multiple seeds
(recommended: 3-5 seeds, report mean ± std). Then write the risk note on deploying a model
trained on synthetic-augmented data — a negative or null result, well-analyzed, is a fully
acceptable and gradeable outcome for this lab.

_Your answer here._

---
*Beacon AI · AIBits Academy — Chapter 8: Generative models I: GANs*